In [1]:
#######################################################################
#######################################################################
##
## This notebook is to help cellpose training
## 
#######################################################################
#######################################################################

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import copy
import tifffile

In [3]:
from skimage.io import imread
from skimage.io import imsave
from skimage import filters
from skimage.filters import gaussian
from skimage import morphology
from skimage import measure
from skimage import io, exposure, img_as_ubyte
from skimage.transform import rescale
from skimage.transform import resize

In [4]:
#from scipy import ndimage as ndi
from skimage import (exposure, feature, filters, io, measure,
                      morphology, restoration, segmentation, transform,
                      util)

In [5]:
import napari
from napari.utils import nbscreenshot
from pyclesperanto_prototype import imshow
import pyclesperanto_prototype as cle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [6]:
cle.available_device_names()
# For 3D processing, powerful graphics
# processing units might be necessary
cle.select_device('TX')

/Users/jingkui.wang/miniforge3/envs/neuron_branching/lib/python3.9/site-packages/pyclesperanto_prototype/_tier0/_device.py:77: UserWarning: No OpenCL device found with TX in their name. Using Apple M1 Max instead.
  warnings.warn(f"No OpenCL device found with {name} in their name. Using {device.name} instead.")


<Apple M1 Max on Platform: Apple (2 refs)>

In [7]:
#import czifile
from aicsimageio import AICSImage
from aicsimageio.writers import OmeTiffWriter

In [56]:
# Independently scale each image to 8-bit for display.
def to_uint8(img):
    img = np.asarray(img, dtype=np.float32)
    low, high = img.min(), img.max()
    if high == low:
        return np.zeros(img.shape, dtype=np.uint8)
    return np.rint((img - low) / (high - low) * 255).astype(np.uint8)

In [8]:
inputDir = "/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1"

fileName = "11d_LNP3_20x_limb1__Airyscan_05.czi"

outDir = inputDir + '/images/'

if not os.path.exists(outDir):
    os.makedirs(outDir)

print(outDir)

/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1/images/


In [9]:
img = AICSImage(os.path.join(inputDir, fileName))

In [10]:
image = img.data #returns 6D STCZYX numpy array

In [11]:
print(image.shape)
print(img.dims)  # returns string "STCZYX"
print(img.shape)  # returns tuple of dimension sizes in STCZYX order
#print(img.physical_pixel_sizes.Z)  # no Z dimension
print(img.physical_pixel_sizes.Y)  # returns the Y dimension pixel size as found in the metadata
print(img.physical_pixel_sizes.X)  # returns the X dimension pixel size as found in the metadata

(1, 4, 1, 18868, 40489)
<Dimensions [T: 1, C: 4, Z: 1, Y: 18868, X: 40489]>
(1, 4, 1, 18868, 40489)
0.061810168997668995
0.061810168997668995


In [12]:
hcr_c1 = img.get_image_data("ZYX", C=0, S=0, T=0) # Kazald1, the signal to quantify
prrxCherry_c2 = img.get_image_data("ZYX", C=1, S=0, T=0) # connective tissue cells, signals in the cytoplasm can be used as cell masks
lnpGFP_c3 = img.get_image_data("ZYX", C=2, S=0, T=0) # also cytoplasm, marking cells with lnp, in blastema there is no lnp-GFP, because no injection in blatema, even in the CSD, blood cell make autofluerence
dapi_c4 =  img.get_image_data("ZYX", C=3, S=0, T=0)# dapi channel

In [13]:
## ignore the z stack channel
print(dapi_c4[0].shape)
print(prrxCherry_c2[0].shape)
hcr = hcr_c1[0]
lnpGFP = lnpGFP_c3[0]
cellmarker = prrxCherry_c2[0]
dapi = dapi_c4[0]

(18868, 40489)
(18868, 40489)


In [14]:
print(fileName.removesuffix('.czi'))
print(inputDir)

11d_LNP3_20x_limb1__Airyscan_05
/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1


In [16]:
## convert 16bit to 8bit to reduce the image size
#cellmarker_rescaled = exposure.rescale_intensity(cellmarker, in_range='image', out_range=(0, 65535))
cellmarker8 = img_as_ubyte(cellmarker)
dapi8 = img_as_ubyte(dapi)
lnpGFP8 = img_as_ubyte(lnpGFP)
print([np.min(cellmarker8), np.max(cellmarker8)])
print([np.min(dapi8), np.max(dapi8)])
print([np.min(lnpGFP8), np.max(lnpGFP8)])

[np.uint8(0), np.uint8(255)]
[np.uint8(0), np.uint8(119)]
[np.uint8(0), np.uint8(250)]


In [17]:
sigma = 1.0
cellmarker8 = gaussian(cellmarker8, sigma=sigma)
dapi8 = gaussian(dapi8, sigma=sigma)
lnpGFP8 = gaussian(lnpGFP8, sigma=sigma)

[np.uint16(0), np.uint16(65535)]
[np.uint16(0), np.uint16(30530)]


In [18]:
print([np.min(cellmarker8), np.max(cellmarker8)])
print([np.min(dapi8), np.max(dapi8)])
print([np.min(lnpGFP8), np.max(lnpGFP8)])

[np.float64(0.0), np.float64(0.9930360634305546)]
[np.float64(0.0), np.float64(0.43198521967084835)]
[np.float64(0.0), np.float64(0.9198420753838111)]


In [19]:
## downsample the images
factor = 0.5  # downsample to 50%
cellmarker8 = rescale(cellmarker8, factor, preserve_range=True, anti_aliasing=True, order=1).astype(cellmarker8.dtype)
dapi8 = rescale(dapi8, factor, preserve_range=True, anti_aliasing = True, order=1).astype(dapi8.dtype)
lnpGFP8 = rescale(lnpGFP8, factor, preserve_range=True, anti_aliasing = True, order=1).astype(lnpGFP8.dtype)
print(cellmarker8.shape)
print(dapi8.shape)

(9434, 20244)
(9434, 20244)


In [21]:
output_file = os.path.join(outDir, fileName.removesuffix('.czi') + "_8bit.tif")
print(output_file)

/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1/images/11d_LNP3_20x_limb1__Airyscan_05_8bit.tif


In [52]:
two_channel_image = np.stack([cellmarker8, dapi8], axis= -1)
#three_channel_image = np.stack([cellmarker8, dapi8], axis= 0).astype(np.float32)
#image_8bit = np.clip(np.rint(image * 255), 0, 255).astype(np.uint8)

In [34]:
#print(three_channel_image.shape)
#print([np.min(three_channel_image), np.max(three_channel_image)])

(3, 9434, 20244)
[np.float32(0.0), np.float32(0.9797044)]


In [36]:
two_channel_image_8bit = np.clip(np.rint(three_channel_image*255), 0, 255).astype(np.uint8)

In [37]:
print(three_channel_image_8bit.shape)
print([np.min(three_channel_image_8bit), np.max(three_channel_image_8bit)])

(3, 9434, 20244)
[np.uint8(0), np.uint8(250)]


In [41]:
output_file = os.path.join(outDir, fileName.removesuffix('.czi') + "_8bit.tif")
print(output_file)

/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1/images/11d_LNP3_20x_limb1__Airyscan_05_8bit_2.tif


In [42]:
tifffile.imwrite(
    output_file,
    three_channel_image_8bit,
    #photometric="rgb"
    imagej=True,
    metadata={"axes": "CYX"}
)

In [44]:
marker_8bit = to_uint8(cellmarker8)
dapi_8bit = to_uint8(dapi8)

In [46]:
print([np.min(marker_8bit), np.max(marker_8bit)])
print([np.min(dapi_8bit), np.max(dapi_8bit)])

[np.uint8(0), np.uint8(255)]
[np.uint8(0), np.uint8(255)]


In [47]:
rgb = np.stack(
    [ marker_8bit,                  # Red: cell marker
      np.zeros_like(marker_8bit),  # Green: unused
      dapi_8bit,                    # Blue: DAPI
    ],
    axis=-1,
)

In [50]:
output_file = os.path.join(outDir, fileName.removesuffix('.czi') + "_8bit_rbg.tif")
print(output_file)

/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1/images/11d_LNP3_20x_limb1__Airyscan_05_8bit_rbg.tif


In [51]:
tifffile.imwrite(
    output_file,
    rgb,
    imagej=True,
    photometric="rgb",
    compression="deflate",
)

In [53]:
two_channel_image = np.stack([marker_8bit, dapi_8bit], axis= -1)
#three_channel_image = np.stack([cellmarker8, dapi8], axis= 0).astype(np.float32)
#image_8bit = np.clip(np.rint(image * 255), 0, 255).astype(np.uint8)

In [54]:
output_file = os.path.join(outDir, fileName.removesuffix('.czi') + "_8bit.tif")
print(output_file)

/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1/images/11d_LNP3_20x_limb1__Airyscan_05_8bit.tif


In [55]:
tifffile.imwrite(
    output_file,
    two_channel_image,
    #photometric="rgb"
    imagej=True,
    metadata={"axes": "CYX"}
)

In [ ]:
### bash process all images
for nm in os.listdir(inputDir):
    #if "C4_CystMask.tiff" in nm:
    if ".czi" in nm:
        print(nm)
        #fileName = nm.replace('_C4_CystMask.tiff','')
        fileName = nm.replace('.czi','')
        
        ## read czi file (https://forum.image.sc/t/reading-czi-file-in-python/39768/3)
        img = AICSImage(os.path.join(inputDir, fileName + '.czi'))
        image = img.data #returns 6D STCZYX numpy array
        
        print(image.shape)
        #print(img.dims)  # returns string "STCZYX"
        #print(img.shape)  # returns tuple of dimension sizes in STCZYX order
        print(img.physical_pixel_sizes.Y)  # returns the Y dimension pixel size as found in the metadata
        print(img.physical_pixel_sizes.X)  # returns the X dimension pixel size as found in the metadata
        
        prrxCherry_c2 = img.get_image_data("ZYX", C=1, S=0, T=0) # connective tissue cells, signals in the cytoplasm can be used as cell masks
        dapi_c4 =  img.get_image_data("ZYX", C=3, S=0, T=0)# dapi channel 
        
        print(dapi_c4[0].shape)
        print(prrxCherry_c2[0].shape)
        
        cellmarker = prrxCherry_c2[0]
        dapi = dapi_c4[0]
        
        ## mild denoising
        sigma = 1.0
        cellmarker = gaussian(cellmarker, sigma=sigma)
        dapi = gaussian(dapi, sigma=sigma)
        
        print([np.min(cellmarker), np.max(cellmarker)])
        print([np.min(dapi), np.max(dapi)])
        
        ## convert 16bit to 8bit to reduce the image size
        #cellmarker_rescaled = exposure.rescale_intensity(cellmarker, in_range='image', out_range=(0, 65535))
        cellmarker8 = img_as_ubyte(cellmarker)
        dapi8 = img_as_ubyte(dapi)
        print([np.min(cellmarker8), np.max(cellmarker8)])
        print([np.min(dapi8), np.max(dapi8)])
        
        ## downsample the images
        factor = 0.5  # downsample to 50%
        cellmarker8 = rescale(cellmarker8, factor, preserve_range=True, anti_aliasing=True, order=1).astype(cellmarker8.dtype)
        dapi8 = rescale(dapi8, factor, preserve_range=True, anti_aliasing = True, order=1).astype(dapi8.dtype)
        
        print(cellmarker8.shape)
        print(dapi8.shape)
        
        marker_8bit = to_uint8(cellmarker8)
        dapi_8bit = to_uint8(dapi8)

        ## save the two channels images
        output_file = os.path.join(outDir, fileName + "_8bit.tif")
        
        print(output_file)
        
        two_channel_image = np.stack([marker_8bit, dapi_8bit], axis= -1)
        # Save as an ImageJ-compatible TIFF, labeling axes as CYX
        tifffile.imwrite(
            output_file,
            two_channel_image,
            imagej=True,
            metadata={"axes": "CYX"}
        )
        

11dpi_LNP2_an2_20x_Airyscan_09.czi
(1, 4, 1, 18877, 68296)
0.061810168997668995
0.061810168997668995
(18877, 68296)
(18877, 68296)
[np.float64(0.0), np.float64(0.9932205968451748)]
[np.float64(0.0), np.float64(0.537209744373853)]
[np.uint8(0), np.uint8(253)]
[np.uint8(0), np.uint8(137)]
(9438, 34148)
(9438, 34148)
/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/axolotl_limb_CSD/HCR_Kazald1/images/11dpi_LNP2_an2_20x_Airyscan_09_8bit.tif
11d_LNP3_20x_limb2_Airyscan_09.czi
(1, 4, 1, 22024, 52847)
0.061810168997668995
0.061810168997668995
(22024, 52847)
(22024, 52847)
[np.float64(0.0), np.float64(1.0)]
[np.float64(0.0), np.float64(0.7255450664265867)]
[np.uint8(0), np.uint8(255)]
[np.uint8(0), np.uint8(185)]
